# Ranked Retrieval with TF-IDF and Cosine Similarity

Si superano i limiti del Boolean Retrieval (dove il risultato è un insieme di documenti senza ordinamento per rilevanza) costruendo un sistema di Ranked Retrieval tramite rappresentazione vettoriale dei documenti, usando TF-IDF per pesare i termini e Cosine Similarity per ordinare i documenti in base alla rilevanza rispetto alla query.

Si evita anche il problema **feast or famine** --> non chiediamo più solo se il documento soddisfa la query, ma quanto è vicino ad essa.

Di nuovo preprocessing, si arriva a un dict tokenized_documents che contiene come chiavi i doc_id e come valori le liste di token. 

Per il VSM useremo una rappresentazione Bag of Words, quindi non consideriamo l'ordine dei termini, ma solo la loro frequenza. 

Per la rappresentazione vettoriale anzitutto è necessario costruire il vocabolario della collezione, cioè l'insieme ordinato dei termini distinti osservati in tutti i documenti preprocessati. Ogni termine del vocabolario sarà infatti una dimensione nell'iperspazio vettoriale.

```python
def build_vocabulary(tokenized_documents):
    vocabulary = list(
        set(
            token
            for tokens in tokenized_documents.values()
            for token in tokens
        )
    )
    return vocabulary
```

Ci servono tf e idf per pesare i termini. Partiamo calcolando df, cioè il numero di documenti in cui compare ciascun termine. 

```python
def build_document_frequency(tokenized_documents):
    df = defaultdict(int)

    for tokens in tokenized_documents.values():
        seen_terms = set(tokens)
        for term in seen_terms:
            df[term] += 1

    return dict(df)
```

Per quel che riguarda tf, si usa la trasformazione logaritmica perché è importante che questo peso cresca con la frequenza del termine, ma non in modo lineare, altrimenti i termini troppo frequenti avrebbero un peso eccessivo. 

In particolare si adotta la seguente formula:
$$tf(t, d) = 1 + \log_{10}(f_{t,d}) \quad \text{se } f_{t,d} > 0, \text{ altrimenti } 0 $$
dove $f_{t,d}$ è la frequenza del termine $t$ nel documento $d$.

```python
def compute_term_frequencies(tokens):
    return Counter(tokens)

def log_tf_weight(tf):
    if tf > 0:
        return 1.0 + math.log10(tf)
    return 0.0
```

Per quanto riguarda idf:
$$idf(t) = \log_{10}\left(\frac{N}{df(t)}\right)$$
dove $N$ è il numero totale di documenti nella collezione e $df(t)$ è il numero di documenti in cui compare il termine $t$.

```python
def compute_idf(term, document_frequency, N):
    df = document_frequency.get(term, 0)

    if df == 0:
        return 0.0

    return math.log10(N / df)
```

Si combinano poi i due pesi per ottenere il peso TF-IDF di ciascun termine in ciascun documento:
$$w_{t,d} = tf(t, d) \cdot idf(t)$$

```python
def compute_tf_idf_weights(tokens, document_frequency, N):
    """
    Calcola i pesi tf-idf di un documento.

    Returns
    -------
    dict[str, float]
        termine -> peso tf-idf
    """
    tf_counter = compute_term_frequencies(tokens)
    weights = {}

    for term, tf in tf_counter.items():
        weights[term] = log_tf_weight(tf) * compute_idf(term, document_frequency, N)

    return weights
```

A questo punto ogni documento può essere visto come un vettore molto grande, dove si ha una dimensione per ogni termine del dizionario, e il peso del termine in quel documento è dato da tf-idf. Si hanno tantissimi zeri nelle dimensioni corrispondenti ai termini che non compaiono in quel documento, quindi si parla di vettori sparsi (sparse vectors) --> è conveniente usare una rappresentazione sparsa, memorizzando solo i termini con peso non nullo.

Per farlo usiamo scriviamo una funzione che restituisce un dizionario che mappa ogni doc_id al suo corrispondente dizionario di pesi tf-idf. 

```python
def build_document_vectors(tokenized_documents, document_frequency):
    """
    Costruisce la rappresentazione sparsa tf-idf di tutti i documenti.

    Returns
    -------
    dict[int, dict[str, float]]
        docID -> {termine: peso tf-idf}
    """
    N = len(tokenized_documents)
    document_vectors = {}

    for doc_id, tokens in tokenized_documents.items():
        document_vectors[doc_id] = compute_tf_idf_weights(tokens, document_frequency, N)

    return document_vectors
```

Nel VSM anche la query viene rappresentata come un vettore nello stesso spazio dei documenti, usando la stessa formula di pesatura tf-idf. In questo modo sia i documenti che la query vivono nello stesso spazio vettoriale e quindi è possibile confrontarli usando una misura di similarità.

Per la query quindi così come visto per i documenti si fa il preprocessing e si calcolano i pesi tf-idf, usando ovviamente come df quello calcolato sui documenti, e come N il numero di documenti nella collezione.

A questo punto si usa la **cosine similarity** per confrontare la query con ciascun documento e ottenere un punteggio di rilevanza. La cosine similarity tra due vettori $q$ e $d$ è definita come:
$$\text{cosine\_similarity}(q, d) = \frac{q \cdot d}{\|q\| \|d\|}$$
dove $q \cdot d$ è il prodotto scalare tra i due vettori e $\|q\|$ e $\|d\|$ sono le loro norme euclidee (somma dei quadrati delle componenti, tutto sotto radice). 

L'uso della cosine è importante perché ci interessa la direzione dei vettori, non la lunghezza assoluta (infatti documenti più lunghi tendono ad avere più termini, ma non vogliamo favorire automaticamente termini più lunghi --> la normalizzazione nella cosine similarity serve proprio a questo, a rendere il punteggio indipendente dalla lunghezza dei documenti).

Nel codice ovviamente si sfrutta la natura sparsa dei vettori e la rispettiva rappresentazione, facendo il prodotto scalare solo sui termini che compaiono sia nella query che nel documento.

```python
def dot_product_sparse(vec1, vec2):
    # Iteriamo sul vettore più piccolo per essere più efficienti.
    if len(vec1) > len(vec2):
        vec1, vec2 = vec2, vec1

    score = 0.0
    for term, weight in vec1.items():
        if term in vec2:
            score += weight * vec2[term]
    return score

def l2_norm_sparse(vec):
    return math.sqrt(sum(weight ** 2 for weight in vec.values()))

def cosine_similarity_sparse(query_vec, doc_vec):
    query_norm = l2_norm_sparse(query_vec)
    doc_norm = l2_norm_sparse(doc_vec)
    # MA VA CALCOLATA LA NORMA OGNI VOLTA??
    if query_norm == 0.0 or doc_norm == 0.0:
        return 0.0
    return dot_product_sparse(query_vec, doc_vec) / (query_norm * doc_norm)
```

Il commento nella funzione cosine_similarity_sparse è importante: **nei sistemi IR reali la norma viene calcolata solo una volta, quando costruiscono l'indice, e la salvano.** In questo modo si evita di ricalcolarla ogni volta che si confronta il documento con una query, rendendo il sistema molto più efficiente. Volendo si potrebbe anche salvare i vettori normalizzati, in modo da dover fare solo il prodotto scalare per calcolare la cosine similarity, ma questo è un dettaglio di implementazione.

```python
# ci si salva un dizionario doc_norms che mappa ogni doc_id alla norma del suo vettore, calcolata una volta per tutte quando si costruisce l'indice
doc_norms[doc_id] = l2_norm_sparse(doc_vector)
# poi durante la query:
doc_norm = doc_norms[doc_id]
```

Ora possiamo finalmente eseguire retrieval ranked:
- preprocessing della query
- costruzione del vettore tf-idf della query
- calcolo della cosine similarity tra la query e ciascun documento
- ordinamento dei documenti in base al punteggio di cosine similarity e restituzione dei top-k risultati più rilevanti.

```python
def rank_documents(query, document_vectors, document_frequency, top_k=10):
    """
    Restituisce i top-k documenti per una query usando cosine similarity.

    Returns
    -------
    tuple
        (processed_query_tokens, ranked_results)

    dove ranked_results è una lista di tuple:
        (doc_id, score)
    """
    N = len(document_vectors)
    processed_query, query_vec = build_query_vector(query, document_frequency, N)

    results = []

    for doc_id, doc_vec in document_vectors.items():
        score = cosine_similarity_sparse(query_vec, doc_vec)
        if score > 0:
            results.append((doc_id, score))

    results.sort(key=lambda x: x[1], reverse=True)

    return processed_query, results[:top_k]
```

## Estensione: body e title con pesi diversi
Nei sistemi reali non tutte le parti di un documento hanno la stessa importanza. 

Ad esempio, un termine presente nel titolo può essere molto informativo, mentre lo stesso termine ma nel body può essere meno importante. Per questo una possibile estensione del modello è quella di assegnare pesi diversi ai termini a seconda della loro posizione nel documento, per poi combinarle.

Anzitutto si estraggono title e body dal documento tramite una funzione apposita.

```python
def build_combined_document_vectors(title_tokens_by_doc, body_tokens_by_doc, alpha_title=2.0, alpha_body=1.0):
    """
    Costruisce vettori documento combinando title e body con pesi diversi.

    Parameters
    ----------
    alpha_title : float
        Peso moltiplicativo per i termini del titolo.
    alpha_body : float
        Peso moltiplicativo per i termini del body.

    Returns
    -------
    tuple
        (combined_vectors, combined_df)
    """
    combined_docs = {}

    for doc_id in body_tokens_by_doc:
        combined_docs[doc_id] = {
            "title": title_tokens_by_doc[doc_id],
            "body": body_tokens_by_doc[doc_id]
        }

    # Per il df consideriamo la presenza del termine nel documento,
    # indipendentemente dalla sezione in cui compare.
    combined_tokenized_documents = {}
    for doc_id in combined_docs:
        combined_tokenized_documents[doc_id] = combined_docs[doc_id]["title"] + combined_docs[doc_id]["body"]

    combined_df = build_document_frequency(combined_tokenized_documents)
    N = len(combined_tokenized_documents)

    combined_vectors = {}

    for doc_id in combined_docs:
        title_weights = compute_tf_idf_weights(combined_docs[doc_id]["title"], combined_df, N)
        body_weights = compute_tf_idf_weights(combined_docs[doc_id]["body"], combined_df, N)

        merged = defaultdict(float)

        for term, weight in title_weights.items():
            merged[term] += alpha_title * weight

        for term, weight in body_weights.items():
            merged[term] += alpha_body * weight

        combined_vectors[doc_id] = dict(merged)

    return combined_vectors, combined_df
```